# Reporting Verification

This notebook reads the generated data-quality outputs, reconciles persisted module KPIs against a fresh calculation, summarises rejection reasons and renders two reporting charts. The committed notebook contains no execution output.


In [ ]:
from pathlib import Path

from IPython.display import display
from matplotlib import pyplot as plt

from reporting import (
    build_rejection_reason_summary,
    build_reporting_summary,
    create_metric_figure,
    load_reporting_inputs,
    reconcile_module_kpis,
)


In [ ]:
DATA_QUALITY_DIR = Path(".ci-output/data-quality")
NOTEBOOK_OUTPUT_DIR = Path(".ci-output/reporting-notebook")
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

inputs = load_reporting_inputs(DATA_QUALITY_DIR)
module_kpis = reconcile_module_kpis(inputs.cleaned, inputs.module_kpis)
rejection_summary = build_rejection_reason_summary(inputs.rejected)
reporting_summary = build_reporting_summary(
    inputs,
    module_kpis,
    rejection_summary,
)


## Verified module KPIs


In [ ]:
display(module_kpis)


## Rejection reason summary


In [ ]:
display(rejection_summary)


## Average score by module


In [ ]:
average_figure = create_metric_figure(
    module_kpis,
    "average_score_percentage",
    "Average score by module",
    "Average score (%)",
)
average_figure.savefig(
    NOTEBOOK_OUTPUT_DIR / "average_score_by_module.svg",
    format="svg",
    metadata={"Date": None},
)
display(average_figure)
plt.close(average_figure)


## Pass rate by module


In [ ]:
pass_rate_figure = create_metric_figure(
    module_kpis,
    "pass_rate_percentage",
    "Pass rate by module",
    "Pass rate (%)",
)
pass_rate_figure.savefig(
    NOTEBOOK_OUTPUT_DIR / "pass_rate_by_module.svg",
    format="svg",
    metadata={"Date": None},
)
display(pass_rate_figure)
plt.close(pass_rate_figure)


## Control totals


In [ ]:
assert reporting_summary["kpi_reconciliation"] == "passed"
assert reporting_summary["module_count"] == 4
assert reporting_summary["result_count"] == 8
assert reporting_summary["rejected_row_count"] == 7
assert reporting_summary["overall_average_score_percentage"] == 70.0
assert reporting_summary["overall_pass_rate_percentage"] == 62.5
assert reporting_summary["rejection_reason_count"] == 7

print("Reporting notebook verification passed.")
reporting_summary
